# CardioCare EDA and Preprocessing

This notebook documents the data understanding work that motivates the reusable preprocessing pipeline in `src/preprocessing.py`. The dataset is the merged UCI Heart Disease processed files, with the original multi-class target converted to binary heart disease status.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.preprocessing import (
    CONTINUOUS_FEATURES,
    FEATURE_COLUMNS,
    build_preprocessor,
    iqr_outlier_summary,
    load_dataset,
    prepare_dataset,
    split_features_target,
    validate_feature_ranges,
)

pd.set_option('display.max_columns', None)

## Load Data

The repository includes `data/heart_disease.csv`. If it is absent, `prepare_dataset()` downloads the four UCI processed files and rebuilds the curated CSV.

In [ ]:
df = prepare_dataset()
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## Target Distribution

The target distribution determines why plain accuracy is not sufficient. Balanced accuracy, recall, precision, and F1 are reported in training so both classes contribute to evaluation.

In [ ]:
df['target'].value_counts(normalize=True).rename('proportion').to_frame()

## Missing Values, Duplicates, and Range Checks

The merged UCI files contain missing values in several columns because not every site recorded every field. The reusable pipeline imputes these values rather than dropping large parts of the merged dataset.

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_rate': df.isna().mean(),
})
missing

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_count

In [ ]:
X, y = split_features_target(df)
validate_feature_ranges(X, raise_on_error=False)

## Outlier Review

Continuous fields are reviewed with boxplots and IQR counts. The production pipeline clips continuous features using IQR bounds fit on the training fold only.

In [ ]:
iqr_outlier_summary(df)

In [ ]:
fig, axes = plt.subplots(1, len(CONTINUOUS_FEATURES), figsize=(16, 3.5))
for ax, column in zip(axes, CONTINUOUS_FEATURES):
    ax.boxplot(df[column].dropna(), vert=True)
    ax.set_title(column)
plt.tight_layout()

## Reusable Preprocessing Pipeline

EDA shows that the dataset has missing clinical fields, heterogeneous coded categorical variables, and continuous outliers. The implementation therefore uses a reusable sklearn `ColumnTransformer`: continuous fields get train-fold IQR clipping, median imputation, and `StandardScaler`; categorical-coded fields get most-frequent imputation and one-hot encoding. The transformer is fit only after the train/test split or inside cross-validation folds, preventing data leakage.

In [ ]:
preprocessor = build_preprocessor()
preprocessor